# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayazahmedbscsf23-art/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [6]:
# Section 1: ranked actions + reason codes
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = Path("work/outputs")
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42

# Load the scored pages from earlier work if the file exists, else build a small demo table.
INPUT = OUT / "scored_pages.csv"
if INPUT.exists():
    df = pd.read_csv(INPUT)
    DATA_SOURCE = "scored_pages.csv"
else:
    rng = np.random.default_rng(SEED)
    n = 200
    position = np.round(rng.uniform(1, 35, n), 1)
    impressions = rng.lognormal(6.5, 1.0, n).astype(int) + 20
    base_ctr = 0.30 * np.exp(-0.18 * (position - 1)) + 0.005
    ctr = np.clip(base_ctr * rng.normal(0.85, 0.30, n), 0.001, 0.6)
    clicks = rng.binomial(impressions, ctr)
    prior_clicks = np.maximum(1, (clicks * rng.normal(1.05, 0.30, n)).astype(int))
    df = pd.DataFrame({
        "content_id": [f"page_{i:03d}" for i in range(n)],
        "impressions": impressions,
        "clicks": clicks,
        "position": position,
        "prior_clicks": prior_clicks,
        "days_since_update": rng.integers(5, 700, n),
        "word_count": rng.integers(250, 2600, n),
    })
    DATA_SOURCE = "synthetic demo table (placeholder, not real data)"
print("data source:", DATA_SOURCE, "| rows:", len(df))

REQUIRED = ["content_id", "impressions", "clicks", "position",
            "prior_clicks", "days_since_update", "word_count"]
missing = [c for c in REQUIRED if c not in df.columns]
assert not missing, f"missing columns: {missing}"

# Derived signals
df["ctr"] = df["clicks"] / df["impressions"].clip(lower=1)
df["expected_ctr"] = 0.30 * np.exp(-0.18 * (df["position"] - 1)) + 0.005   # simple position benchmark
df["ctr_gap"] = df["expected_ctr"] - df["ctr"]
df["click_trend"] = (df["clicks"] - df["prior_clicks"]) / df["prior_clicks"].clip(lower=1)

# Ordered rules: first match wins. Each rule = (action, reason_code, plain words, test)
RULES = [
    ("REFRESH_CONTENT", "R1_DECAY",
     "Clicks fell 25%+ vs prior period and page is 180+ days old",
     lambda d: (d["click_trend"] <= -0.25) & (d["days_since_update"] > 180)),
    ("REWRITE_TITLE_META", "R2_LOW_CTR_TOP10",
     "On page one but CTR is well below the benchmark for its position",
     lambda d: (d["position"] <= 10) & (d["ctr_gap"] > 0.02)),
    ("PUSH_TO_PAGE_ONE", "R3_NEAR_PAGE_ONE",
     "Ranks 11-20 with solid impressions; small gains could reach page one",
     lambda d: d["position"].between(10.01, 20) & (d["impressions"] >= 500)),
    ("EXPAND_THIN_CONTENT", "R4_THIN_OFF_PAGE_ONE",
     "Under 600 words and ranking off page one",
     lambda d: (d["word_count"] < 600) & (d["position"] > 10)),
]

df["action"] = "MONITOR"
df["reason_code"] = "R0_NO_RULE"
df["reason_text"] = "No rule fired; keep watching"
assigned = pd.Series(False, index=df.index)
for action, code, words, test in RULES:
    hit = test(df) & ~assigned
    df.loc[hit, ["action", "reason_code", "reason_text"]] = [action, code, words]
    assigned |= hit

# Directional click-gain estimate and confidence (more impressions = more trust)
gain_rate = {"REFRESH_CONTENT": 0.25, "REWRITE_TITLE_META": 1.0,
             "PUSH_TO_PAGE_ONE": 0.5, "EXPAND_THIN_CONTENT": 0.3, "MONITOR": 0.0}
df["est_click_gain"] = np.where(
    df["action"] == "REWRITE_TITLE_META",
    df["impressions"] * df["ctr_gap"].clip(lower=0) * gain_rate["REWRITE_TITLE_META"],
    np.where(df["action"] == "PUSH_TO_PAGE_ONE",
             df["impressions"] * df["expected_ctr"] * gain_rate["PUSH_TO_PAGE_ONE"],
             df["clicks"] * df["action"].map(gain_rate)))
df["confidence"] = (df["impressions"] / 1000).clip(upper=1.0).round(2)
df["priority_score"] = (df["est_click_gain"] * df["confidence"]).round(2)

queue = (df[df["action"] != "MONITOR"]
         .sort_values("priority_score", ascending=False)
         .reset_index(drop=True))
queue.insert(0, "rank", queue.index + 1)

print("actions in queue:", len(queue))
print(queue["action"].value_counts().to_string())
queue[["rank", "content_id", "action", "reason_code", "priority_score", "confidence"]].head(10)

data source: synthetic demo table (placeholder, not real data) | rows: 200
actions in queue: 101
action
PUSH_TO_PAGE_ONE       33
REWRITE_TITLE_META     31
REFRESH_CONTENT        20
EXPAND_THIN_CONTENT    17


,rank,content_id,action,reason_code,priority_score,confidence
0,1,page_051,REWRITE_TITLE_META,R2_LOW_CTR_TOP10,328.32,1.00
1,2,page_124,REWRITE_TITLE_META,R2_LOW_CTR_TOP10,161.07,1.00
2,3,page_068,REWRITE_TITLE_META,R2_LOW_CTR_TOP10,140.31,1.00
3,4,page_114,REWRITE_TITLE_META,R2_LOW_CTR_TOP10,136.89,1.00
4,5,page_135,REWRITE_TITLE_META,R2_LOW_CTR_TOP10,136.37,1.00
5,6,page_175,REWRITE_TITLE_META,R2_LOW_CTR_TOP10,95.52,1.00
6,7,page_198,REWRITE_TITLE_META,R2_LOW_CTR_TOP10,77.14,1.00
7,8,page_108,REWRITE_TITLE_META,R2_LOW_CTR_TOP10,69.16,0.64
8,9,page_096,PUSH_TO_PAGE_ONE,R3_NEAR_PAGE_ONE,55.10,1.00
9,10,page_160,REWRITE_TITLE_META,R2_LOW_CTR_TOP10,54.05,0.86


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [7]:
# Section 2: intended use and limits, written as checks the queue must pass
INTENDED_USE = {
    "users": "content / SEO analysts deciding what to work on next",
    "purpose": "decision-support ranking of pages; directional, not a forecast",
    "not_for": ["automatic publishing or edits", "performance reviews of people",
                "pages outside the validity bounds below"],
}

# Validity bounds: outside these the rules and gain estimates are not trusted
LIMITS = {"min_impressions": 100, "position_min": 1, "position_max": 50, "min_rows": 30}

df["in_scope"] = (
    (df["impressions"] >= LIMITS["min_impressions"])
    & df["position"].between(LIMITS["position_min"], LIMITS["position_max"])
)
queue["in_scope"] = queue["content_id"].map(df.set_index("content_id")["in_scope"])

scope_rate = df["in_scope"].mean()
queue_in_scope = queue[queue["in_scope"]].reset_index(drop=True)
queue_in_scope["rank"] = queue_in_scope.index + 1

print(f"pages in scope: {df['in_scope'].sum()} of {len(df)} ({scope_rate:.0%})")
print(f"queue rows kept: {len(queue_in_scope)} of {len(queue)}")
print("score caveat: est_click_gain is an observed-pattern estimate, directional only")

# Hard checks: stop the run if the table is too small to rank honestly
assert len(df) >= LIMITS["min_rows"], "too few rows to rank"
assert queue_in_scope["priority_score"].is_monotonic_decreasing, "queue not sorted"
assert queue_in_scope["reason_code"].notna().all(), "every action needs a reason code"
pd.Series(INTENDED_USE["not_for"], name="not_for").to_frame()

pages in scope: 196 of 200 (98%)
queue rows kept: 98 of 101
score caveat: est_click_gain is an observed-pattern estimate, directional only


,not_for
0,automatic publishing or edits
1,performance reviews of people
2,pages outside the validity bounds below


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [8]:
# Section 3: human review gate + no-go list
# Actions the system may suggest but a person must always confirm
REVIEW_REQUIRED_ACTIONS = {"REFRESH_CONTENT", "REWRITE_TITLE_META",
                           "PUSH_TO_PAGE_ONE", "EXPAND_THIN_CONTENT"}

# Actions that must never be produced or automated by this playbook
NO_GO = ["DELETE_PAGE", "REDIRECT_OR_MERGE", "NOINDEX", "CHANGE_CANONICAL",
         "AUTO_PUBLISH", "AUTO_REWRITE_LIVE_COPY"]

CHECKLIST = [
    "Open the page and confirm the problem the reason code describes is real",
    "Check search intent still matches the page",
    "Check for a known cause (seasonality, site change, tracking gap)",
    "Confirm the page is not legal, compliance, or brand-sensitive",
    "Edit and publish by a person only; log who approved and when",
]

HIGH_IMPACT_Q = 0.80
high_cut = queue_in_scope["priority_score"].quantile(HIGH_IMPACT_Q)

def review_flags(row):
    flags = []
    if row["action"] in REVIEW_REQUIRED_ACTIONS:
        flags.append("human_confirm")
    if row["confidence"] < 0.5:
        flags.append("low_confidence")
    if row["priority_score"] >= high_cut:
        flags.append("high_impact_second_look")
    return "|".join(flags)

queue_in_scope["review_flags"] = queue_in_scope.apply(review_flags, axis=1)
queue_in_scope["auto_allowed"] = False   # nothing in this queue is auto-executed

# Guard: the queue must never contain a no-go action
bad = set(queue_in_scope["action"]) & set(NO_GO)
assert not bad, f"no-go actions found: {bad}"
assert not queue_in_scope["auto_allowed"].any()

print("no-go list:", ", ".join(NO_GO))
print("rows needing a low-confidence look:",
      queue_in_scope["review_flags"].str.contains("low_confidence").sum())
print("rows flagged high impact:",
      queue_in_scope["review_flags"].str.contains("high_impact_second_look").sum())
queue_in_scope[["rank", "content_id", "action", "review_flags", "auto_allowed"]].head(10)

no-go list: DELETE_PAGE, REDIRECT_OR_MERGE, NOINDEX, CHANGE_CANONICAL, AUTO_PUBLISH, AUTO_REWRITE_LIVE_COPY
rows needing a low-confidence look: 22
rows flagged high impact: 20


,rank,content_id,action,review_flags,auto_allowed
0,1,page_051,REWRITE_TITLE_META,human_confirm|high_impact_second_look,False
1,2,page_124,REWRITE_TITLE_META,human_confirm|high_impact_second_look,False
2,3,page_068,REWRITE_TITLE_META,human_confirm|high_impact_second_look,False
3,4,page_114,REWRITE_TITLE_META,human_confirm|high_impact_second_look,False
4,5,page_135,REWRITE_TITLE_META,human_confirm|high_impact_second_look,False
5,6,page_175,REWRITE_TITLE_META,human_confirm|high_impact_second_look,False
6,7,page_198,REWRITE_TITLE_META,human_confirm|high_impact_second_look,False
7,8,page_108,REWRITE_TITLE_META,human_confirm|high_impact_second_look,False
8,9,page_096,PUSH_TO_PAGE_ONE,human_confirm|high_impact_second_look,False
9,10,page_160,REWRITE_TITLE_META,human_confirm|high_impact_second_look,False


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [9]:
# Section 4: monitoring and retrain triggers
def psi(base, recent, bins=10):
    # Population Stability Index on quantile bins of the baseline
    edges = np.unique(np.quantile(base, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    b = np.histogram(base, edges)[0] / len(base)
    r = np.histogram(recent, edges)[0] / len(recent)
    b, r = np.clip(b, 1e-4, None), np.clip(r, 1e-4, None)
    return float(np.sum((r - b) * np.log(r / b)))

# Recent window: use a real file if present, else a mildly shifted demo copy
RECENT = OUT / "scored_pages_recent.csv"
if RECENT.exists():
    recent = pd.read_csv(RECENT)
    recent["ctr"] = recent["clicks"] / recent["impressions"].clip(lower=1)
else:
    rng2 = np.random.default_rng(SEED + 1)
    recent = df[REQUIRED + ["ctr"]].copy()
    recent["position"] = (recent["position"] + rng2.normal(0.5, 1.5, len(recent))).clip(1, 50)
    recent["ctr"] = (recent["ctr"] * rng2.normal(0.97, 0.15, len(recent))).clip(0.001, 0.6)
    recent["impressions"] = (recent["impressions"] * rng2.normal(1.0, 0.1, len(recent))).astype(int).clip(lower=1)

TRIGGERS = {"psi_warn": 0.10, "psi_retrain": 0.25,
            "top20_overlap_min": 0.50, "empty_queue_share_max": 0.90}

checks = []
for col in ["position", "ctr", "impressions"]:
    v = psi(df[col].to_numpy(), recent[col].to_numpy())
    status = "RETRAIN" if v >= TRIGGERS["psi_retrain"] else "WARN" if v >= TRIGGERS["psi_warn"] else "OK"
    checks.append({"check": f"psi_{col}", "value": round(v, 3), "status": status})

# Does the top of the queue still look the same when rules run on recent data?
recent_ids = set(recent.sort_values("impressions", ascending=False)["content_id"].head(20)) \
    if "content_id" in recent.columns else set()
top_ids = set(queue_in_scope["content_id"].head(20))
overlap = len(top_ids & recent_ids) / max(1, len(top_ids))
checks.append({"check": "top20_overlap_vs_recent_traffic", "value": round(overlap, 2),
               "status": "OK" if overlap >= TRIGGERS["top20_overlap_min"] else "REVIEW"})

monitor = pd.DataFrame(checks)
needs_retrain = (monitor["status"] == "RETRAIN").any()
print("retrain trigger fired:", needs_retrain)
print("note: recent window is", "a real file" if RECENT.exists() else "a demo shift (placeholder)")
monitor

retrain trigger fired: False
note: recent window is a demo shift (placeholder)


,check,value,status
0,psi_position,0.011,OK
1,psi_ctr,0.027,OK
2,psi_impressions,0.013,OK
3,top20_overlap_vs_recent_traffic,0.200,REVIEW


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [10]:
# Section 5: exports for the paper
queue_cols = ["rank", "content_id", "action", "reason_code", "reason_text",
              "impressions", "clicks", "position", "ctr", "expected_ctr",
              "est_click_gain", "confidence", "priority_score",
              "review_flags", "auto_allowed"]
queue_out = queue_in_scope[queue_cols].round(4)
queue_out.to_csv(OUT / "action_queue.csv", index=False)
monitor.to_csv(OUT / "monitoring_checks.csv", index=False)

playbook_meta = {
    "data_source": DATA_SOURCE,
    "rows_scored": int(len(df)),
    "queue_rows": int(len(queue_out)),
    "limits": LIMITS,
    "no_go": NO_GO,
    "review_checklist": CHECKLIST,
    "triggers": TRIGGERS,
    "wording": "decision-support, directional estimates, observed patterns",
}
with open(OUT / "playbook_meta.json", "w") as f:
    json.dump(playbook_meta, f, indent=2)

# Figure 1: top 10 actions by priority score
top = queue_out.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top["content_id"] + " | " + top["action"], top["priority_score"], color="#4c78a8")
ax.set_xlabel("priority score (directional)")
ax.set_title("Top 10 queued actions")
fig.tight_layout()
fig.savefig(OUT / "fig_top_actions.png", dpi=150)
plt.close(fig)

# Figure 2: actions by type
fig, ax = plt.subplots(figsize=(6, 3.5))
queue_out["action"].value_counts().plot.bar(ax=ax, color="#72b7b2")
ax.set_ylabel("pages")
ax.set_title("Queue by action type")
fig.tight_layout()
fig.savefig(OUT / "fig_action_mix.png", dpi=150)
plt.close(fig)

# Confirm every export landed and is non-empty
expected = ["action_queue.csv", "monitoring_checks.csv", "playbook_meta.json",
            "fig_top_actions.png", "fig_action_mix.png"]
for name in expected:
    p = OUT / name
    assert p.exists() and p.stat().st_size > 0, f"missing export: {name}"
    print("ok", p, p.stat().st_size, "bytes")

ok work/outputs/action_queue.csv 17537 bytes
ok work/outputs/monitoring_checks.csv 126 bytes
ok work/outputs/playbook_meta.json 948 bytes
ok work/outputs/fig_top_actions.png 68407 bytes
ok work/outputs/fig_action_mix.png 28891 bytes


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.